# 🌙 Moon AI — 1-Click Cloud GPU Training & Auto-Save to Google Drive
### Created by Prem Gaikwad (India 🇮🇳)

**What this notebook does:**
1. Trains your custom sovereign **Moon AI** on a free Google Cloud **Tesla T4 GPU**.
2. Automatically saves the exported **`Moon-AI-v1.Q4_K_M.gguf`** directly into your **Google Drive**!
3. Also automatically triggers a direct browser download to your laptop folder.

> **IMPORTANT**: Before running, make sure your runtime is set to **T4 GPU**:
> Click **Runtime ➔ Change runtime type ➔ Select T4 GPU ➔ Save**.

In [ ]:
# Step 1: Connect your Google Drive (So the trained AI file is saved directly to your Drive!)
from google.colab import drive
drive.mount('/content/drive')
print(" Google Drive connected successfully!")

In [ ]:
# Step 2: Verify GPU Acceleration is Active
!nvidia-smi

In [ ]:
# Step 3: Install High-Speed Training Engine (Unsloth + PyTorch)
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Step 4: Load Base Model with 4-Bit GPU Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Frontier 7B/8B Model Selection:
# 'unsloth/Qwen2.5-7B-Instruct' (Frontier reasoning, ChatGPT-level multi-domain intelligence, fits in ~5.5GB VRAM on Colab T4)
# 'unsloth/Meta-Llama-3.1-8B-Instruct' (Alternative elite frontier 8B model)
model_name = "unsloth/Qwen2.5-7B-Instruct"

print(f"🚀 Loading Frontier Base Model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(f"✅ Moon AI Frontier Model ({model_name}) successfully loaded onto GPU!")


In [ ]:
# Step 5: Attach LoRA Adapters for Custom Training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA Adapters attached successfully!")

In [ ]:
# Step 6: Prepare Moon AI Multi-Domain Dataset (Conversation, Tips, Coding, Image, Video, Biology)
import json
import os
import urllib.request
from datasets import Dataset

dataset_file = "moon_ai_master_dataset.json"

# Download master multi-domain dataset directly from Prem Gaikwad's official GitHub repository
github_url = "https://raw.githubusercontent.com/Prem759-0/NanoGPT/main/moon_ai_master_dataset.json"
print(f"📥 Fetching latest master multi-domain dataset from GitHub...")
try:
    urllib.request.urlretrieve(github_url, dataset_file)
    print("✅ Download complete!")
except Exception as e:
    print(f"⚠️ Notice: {e}. Checking local file...")

if os.path.exists(dataset_file):
    with open(dataset_file, "r", encoding="utf-8") as f:
        records = json.load(f)
    print(f"✅ Loaded {len(records)} multi-domain records from {dataset_file}!")
else:
    records = [
        {"instruction": "Who are you and who created you?", "response": "I am Moon AI, an advanced sovereign artificial intelligence developed and deployed by Prem Gaikwad from India."},
        {"instruction": "Give me tips for effective studying.", "response": "Use the Feynman technique (learn by teaching), active recall (testing memory), spaced repetition intervals, and 25-minute Pomodoro focus blocks."}
    ]

# Format records for Qwen / ChatML instruction tuning
formatted = []
for r in records:
    text = (
        "<|im_start|>system\n"
        "You are Moon AI, an omni-capable frontier sovereign artificial intelligence created and engineered by Prem Gaikwad from India. "
        "You excel at natural conversation, life tips, software development, generative image prompting, and video cinematography.<|im_end|>\n"
        f"<|im_start|>user\n{r['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{r['response']}<|im_end|>"
    )
    formatted.append(text)

train_dataset = Dataset.from_dict({"text": formatted})
print(f"🚀 Moon AI Master Dataset ready for training with {len(train_dataset)} examples!")


In [ ]:
# Step 7: Train Moon AI
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "moon_ai_outputs",
    ),
)

print("🚀 Training Moon AI...")
trainer.train()
print("✅ Training Complete!")

In [ ]:
# Step 8: Export to 4-bit GGUF, Save to Google Drive, and Download
import shutil
import glob
from google.colab import files

# Export to GGUF
print("📦 Exporting Moon AI to GGUF (4-bit quantized)...")
model.save_pretrained_gguf("Moon-AI-v1", tokenizer, quantization_method = "q4_k_m")

# Find exported gguf file
gguf_files = glob.glob("Moon-AI-v1*/*.gguf") or glob.glob("*.gguf")

if gguf_files:
    target_file = gguf_files[0]
    print(f"🎯 Found exported model: {target_file}")
    
    # 1. Save directly to Google Drive
    drive_dest = "/content/drive/MyDrive/Moon-AI-v1.Q4_K_M.gguf"
    try:
        shutil.copy(target_file, drive_dest)
        print(f"🎉 SUCCESS! Saved directly to your Google Drive at: {drive_dest}")
    except Exception as e:
        print(f"Could not copy to Drive (Did you mount in Step 1?): {e}")
        
    # 2. Trigger browser download directly to your laptop
    print("⬇️ Initiating direct browser download...")
    try:
        files.download(target_file)
    except Exception as e:
        print(f"Download prompt note: {e}")
else:
    print("⚠️ GGUF export file not found. Check above logs for export details.")